# 03 · Data wrangling — defining the landing label

**Project context and attribution.** Developed for the IBM Applied Data Science Capstone (Coursera / IBM Skills Network). IBM supplied the original educational project framework and public teaching datasets; this is a reorganized, independently worded notebook incorporating the student's analysis and subsequent code corrections, with AI-assisted review. The archived teaching data are historical and are **not** a current account of SpaceX launches.

**Run environment:** Python 3 in VS Code's Jupyter notebook extension. Install the project dependencies outside the notebook using the repository's `requirements.txt`. There are no `pip`, `piplite`, JavaScript `fetch`, or notebook-environment-specific installation cells here.

## Aim and data source

Analyze the provided historical SpaceX part-1 dataset and define a binary outcome for first-stage landing attempts. The course labels **`True Ocean` as class 1** even though an ocean landing is not the same thing as successfully recovering a reusable booster. We retain this convention for compatibility with the course, and state it openly. By default use IBM's frozen dataset to match the published capstone figures; a switch below lets you work with your own API output.

In [ ]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IBM_DATASET_PART_1 = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv"
USE_LOCAL_API_OUTPUT = False  # Change to True after running notebook 01, if desired
local_input = OUTPUT_DIR / "dataset_part_1.csv"

if USE_LOCAL_API_OUTPUT:
    if not local_input.exists():
        raise FileNotFoundError(f"Run notebook 01 to create {local_input}, or set USE_LOCAL_API_OUTPUT=False")
    df = pd.read_csv(local_input)
    print("Using your API-derived dataset:", local_input)
else:
    df = pd.read_csv(IBM_DATASET_PART_1)
    print("Using IBM's fixed historical dataset (compatible with course reference).")

print("Shape:", df.shape)
df.head()

## Explore missingness and categories

In [ ]:
print("Missing fields (% of rows):")
print((df.isna().mean() * 100).round(1).sort_values(ascending=False))
print("\nLaunch sites:")
print(df["LaunchSite"].value_counts(dropna=False))
print("\nOrbits:")
print(df["Orbit"].value_counts(dropna=False))
print("\nLanding outcomes:")
print(df["Outcome"].value_counts(dropna=False))

## Construct an explicit target

The original exercise selected `bad_outcomes` by **positions** in a `value_counts()` result. That can break if frequencies change. Instead, encode success by the documented `True` prefix (`True ASDS`, `True RTLS`, `True Ocean`), and other historically observed statuses as class 0. Unknown labels trigger an error rather than quietly receiving the wrong class.

In [ ]:
known_success = {"True ASDS", "True RTLS", "True Ocean"}
known_failure = {"None None", "False ASDS", "False Ocean", "None ASDS", "False RTLS"}

raw_outcomes = df["Outcome"].astype("string").str.strip()
unknown = set(raw_outcomes.dropna().unique()) - known_success - known_failure
if raw_outcomes.isna().any():
    unknown.add("<missing>")
if unknown:
    raise ValueError(f"Unrecognized landing outcomes; review before labeling: {sorted(unknown)}")

df = df.copy()
df["Class"] = raw_outcomes.isin(known_success).astype(int)
print("Class 1 = recorded True landing outcome (includes True Ocean)")
print("Class 0 = other recorded landing outcome")
print("\nCounts:")
print(df["Class"].value_counts().sort_index())
print("\nSuccess fraction in course labeling:", f"{df['Class'].mean():.1%}")
df[["FlightNumber", "Outcome", "Class"]].head(12)

## Cross-check and export

The **sample size and success proportion can differ** between the 90-row course wrangling dataset and the separate dashboard dataset. Do not combine their percentages without checking the source and labeling conventions.

In [ ]:
assert df["Class"].isin([0, 1]).all()
assert df["Class"].notna().all()
output_file = OUTPUT_DIR / "dataset_part_2.csv"
df.to_csv(output_file, index=False)
print("Saved:", output_file.resolve())
print("Rows:", len(df), "Columns:", df.shape[1])
print("Class counts:", df["Class"].value_counts().to_dict())

## What this notebook demonstrates

- Missing-data inspection and categorical summaries
- Explicit, reproducible, validated target engineering
- Understanding the difference between **recorded touchdown** and **operational booster recovery**

**Credit:** IBM Skills Network / Coursera Applied Data Science Capstone (original data wrangling framework and class-label convention); code clarified for reproducible local execution.